# Learned Stage-Wise Curvature Tuning on CUB-200

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the CUB-200 bird classification dataset using an ImageNet-pretrained ResNet-18.

CUB-200 is a fine-grained classification task containing many visually similar bird categories. This makes it useful for testing whether stage-wise curvature adaptation can improve the representation of subtle visual differences between closely related classes.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to examine whether additional optimization improves classification performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [32]:
!nvidia-smi

Thu Oct  1 08:08:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             86W /   70W |    2597MiB /  15360MiB |     49%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 563, done.
remote: Counting objects: 100% (235/235), done.
remote: Compressing objects: 100% (112/112), done.
remote: Total 563 (delta 147), reused 179 (delta 102), pack-reused 328 (from 1)
Receiving objects: 100% (563/563), 6.52 MiB | 4.99 MiB/s, done.
Resolving deltas: 100% (278/278), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
c0c969f (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated OCTMNIST notebook


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [2]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 1.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 62.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 77.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## CUB 200 Dataset

Before running the experiments, the CUB 200 data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_cub200",
    train_batch_size=32,
    test_batch_size=800,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md: 0.00B [00:00, ?B/s]

cub200.py: 0.00B [00:00, ?B/s]

/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:55: SyntaxWarning: invalid escape sequence '\s'
  images_df = pd.read_csv(image_paths_path, sep='\s+', header=None, names=["image_id", "file_path"])
/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:56: SyntaxWarning: invalid escape sequence '\s'
  labels_df = pd.read_csv(image_labels_path, sep='\s+', header=None, names=["image_id", "label"])
/root/.cache/huggingface/modules/datasets_modules/datasets/randall-lab--cub200/f7aadd51b57da4b2864cf4d2a50611c450decb80865336428f3544a65bed6d70/cub200.py:57: SyntaxWarning: invalid escape sequence '\s'
  split_df = pd.read_csv(train_test_split_path, sep='\s+', header=None, names=["image_id", "is_training"])


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train samples: 4796
Validation samples: 1198
Test samples: 5794
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([106, 123, 110,  22,  52, 130, 136, 163,  97,  37])


# ResNet-18 Experiments

ResNet-18 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Arabic Digits dataset split, experimental seeds, and downstream training protocol.

## ResNet-18 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-18 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [34]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 08:08:44.365 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed42_epochs20.log
2026-10-01 08:08:44.369 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 08:10:10.194 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 08:10:10.217 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 08:10:10.217 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 08:11:01.460 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.639930, Accuracy: 3.12%
2026-10-01 08:11:01.867 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.213926, Accuracy: 2.07%
2026-10-01 08:11:02.274 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.125660, Val Accuracy: 13.19%
2026-10-01 08:11:02.275 | INFO     | train:linear_probe:190 - New best validation accuracy: 13.19 at epoch 1
2026-10-01 08:11:02.338 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.969887

In [35]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 08:49:14.189 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed43_epochs20.log
2026-10-01 08:49:14.192 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 08:50:36.728 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 08:50:36.744 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 08:50:36.744 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 08:51:29.683 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.576430, Accuracy: 0.00%
2026-10-01 08:51:30.105 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.249439, Accuracy: 1.64%
2026-10-01 08:51:30.483 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.096629, Val Accuracy: 16.86%
2026-10-01 08:51:30.484 | INFO     | train:linear_probe:190 - New best validation accuracy: 16.86 at epoch 1
2026-10-01 08:51:30.561 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.663646

In [36]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-01 09:31:33.217 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed44_epochs20.log
2026-10-01 09:31:33.220 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 09:33:12.621 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 09:33:12.645 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 09:33:12.646 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 09:34:13.410 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.421100, Accuracy: 0.00%
2026-10-01 09:34:13.792 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.247695, Accuracy: 2.04%
2026-10-01 09:34:14.273 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.170447, Val Accuracy: 15.69%
2026-10-01 09:34:14.275 | INFO     | train:linear_probe:190 - New best validation accuracy: 15.69 at epoch 1
2026-10-01 09:34:14.381 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.146861

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [37]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 10:17:53.961 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 10:17:54.063 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 10:19:29.647 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 10:19:29.648 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 10:19:29.694 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 10:19:29.695 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 10:19:29.695 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 10:19:31.313 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.458526, Accuracy: 0.00%
2026-10-01 10:20:03.632 | INFO     | train:train_epoch:47 - 

In [38]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 10:41:32.777 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 10:41:32.851 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 10:42:58.333 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 10:42:58.333 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 10:42:58.374 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 10:42:58.375 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 10:42:58.376 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 10:43:00.374 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.347373, Accuracy: 3.12%
2026-10-01 10:43:31.594 | INFO     | train:train_epoch:47 - 

In [39]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 11:05:07.277 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-01 11:05:07.384 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:06:41.471 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 11:06:41.471 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:06:41.503 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 11:06:41.504 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:06:41.504 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 11:06:43.313 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.613837, Accuracy: 0.00%
2026-10-01 11:07:16.391 | INFO     | train:train_epoch:47 - 

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [40]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 11:28:16.564 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed42_epochs20_initrandom_betalr0.01.log
2026-10-01 11:28:16.676 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:29:51.208 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 11:29:51.209 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:29:51.252 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 11:29:51.253 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:29:51.253 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 11:29:53.151 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.466943, Accuracy: 0.00%
2026-10-01 11:30:26.137 | INFO     | train:train

In [41]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 11:51:42.529 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed43_epochs20_initrandom_betalr0.01.log
2026-10-01 11:51:42.621 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 11:53:19.503 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 11:53:19.504 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 11:53:19.545 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-01 11:53:19.545 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 11:53:19.546 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 11:53:21.471 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.344838, Accuracy: 0.00%
2026-10-01 11:53:54.796 | INFO     | train:tra

In [42]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-01 12:12:40.665 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed44_epochs20_initrandom_betalr0.01.log
2026-10-01 12:12:40.726 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 12:13:43.626 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 12:13:43.627 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 12:13:43.659 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-01 12:13:43.659 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 12:13:43.660 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 12:13:44.905 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.611069, Accuracy: 0.00%
2026-10-01 12:14:04.865 | INFO     | train:tra

# ResNet-18 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 20:13:46.930 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed42_epochs30.log
2026-10-01 20:13:46.933 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 20:14:55.012 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 20:14:55.029 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 20:14:55.030 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 20:15:43.556 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.639930, Accuracy: 3.12%
2026-10-01 20:15:43.941 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.213926, Accuracy: 2.07%
2026-10-01 20:15:44.319 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.125660, Val Accuracy: 13.19%
2026-10-01 20:15:44.321 | INFO     | train:linear_probe:190 - New best validation accuracy: 13.19 at epoch 1
2026-10-01 20:15:44.390 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.969887

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 21:16:08.295 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed43_epochs30.log
2026-10-01 21:16:08.297 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 21:17:18.920 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 21:17:18.942 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 21:17:18.942 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 21:18:11.952 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.576430, Accuracy: 0.00%
2026-10-01 21:18:12.384 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.249439, Accuracy: 1.64%
2026-10-01 21:18:12.848 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.096629, Val Accuracy: 16.86%
2026-10-01 21:18:12.850 | INFO     | train:linear_probe:190 - New best validation accuracy: 16.86 at epoch 1
2026-10-01 21:18:12.943 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.663646

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-01 21:58:28.035 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_cub200_resnet18_seed44_epochs30.log
2026-10-01 21:58:28.038 | INFO     | __main__:main:78 - Running on cuda
2026-10-01 21:59:37.315 | INFO     | __main__:main:105 - Testing baseline...
2026-10-01 21:59:37.334 | INFO     | __main__:main:116 - Number of trainable parameters: 102600
2026-10-01 21:59:37.334 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-01 22:00:25.838 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.421100, Accuracy: 0.00%
2026-10-01 22:00:26.187 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 5.247695, Accuracy: 2.04%
2026-10-01 22:00:26.556 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.170447, Val Accuracy: 15.69%
2026-10-01 22:00:26.557 | INFO     | train:linear_probe:190 - New best validation accuracy: 15.69 at epoch 1
2026-10-01 22:00:26.620 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.146861

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 22:38:31.802 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 22:38:31.875 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 22:39:48.715 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 22:39:48.716 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 22:39:48.750 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 22:39:48.751 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 22:39:48.751 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 22:39:50.589 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.458526, Accuracy: 0.00%
2026-10-01 22:40:17.236 | INFO     | train:train_epoch:47 - 

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 23:04:40.932 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 23:04:41.018 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 23:05:51.518 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 23:05:51.518 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 23:05:51.559 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 23:05:51.560 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 23:05:51.560 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 23:05:52.971 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.347373, Accuracy: 3.12%
2026-10-01 23:06:14.845 | INFO     | train:train_epoch:47 - 

In [7]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 23:28:37.139 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-01 23:28:37.192 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 23:29:33.450 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-01 23:29:33.450 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 23:29:33.475 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-01 23:29:33.476 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 23:29:33.476 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 23:29:34.581 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.613837, Accuracy: 0.00%
2026-10-01 23:29:52.808 | INFO     | train:train_epoch:47 - 

In [8]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-01 23:47:47.515 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed42_epochs30_initrandom_betalr0.01.log
2026-10-01 23:47:47.570 | INFO     | __main__:main:302 - Running on cuda
2026-10-01 23:48:57.208 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-01 23:48:57.209 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-01 23:48:57.237 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-01 23:48:57.238 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-01 23:48:57.238 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-01 23:48:58.728 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.466943, Accuracy: 0.00%
2026-10-01 23:49:25.942 | INFO     | train:train

In [9]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 00:14:22.320 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed43_epochs30_initrandom_betalr0.01.log
2026-10-02 00:14:22.412 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 00:15:38.332 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 00:15:38.332 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 00:15:38.363 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-02 00:15:38.364 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 00:15:38.364 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-02 00:15:39.767 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.344838, Accuracy: 0.00%
2026-10-02 00:16:02.472 | INFO     | train:tra

In [10]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds cub200 \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-02 00:40:45.457 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_cub200_resnet18_seed44_epochs30_initrandom_betalr0.01.log
2026-10-02 00:40:45.531 | INFO     | __main__:main:302 - Running on cuda
2026-10-02 00:41:58.109 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-02 00:41:58.110 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
2026-10-02 00:41:58.137 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-02 00:41:58.137 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-02 00:41:58.138 | INFO     | __main__:main:439 - Total trainable parameters: 102604
2026-10-02 00:41:59.696 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 5.611069, Accuracy: 0.00%
2026-10-02 00:42:27.118 | INFO     | train:tra